# Embeddings de palabras

Para aprender espacios de *embeddings* para tareas especificas *keras* facilita la capa `Embedding`.

In [1]:
from keras.layers import Embedding

embedding_layer = Embedding(1000, 64) # 1000 posibles tokens (1+max_word_idx), 64 = dimension del embedding

Se puede entender la capa `Embedding` como un diccionario que mapea indices de palabras a vectores densos. 

La capa recibe tensores 2D (samples, sequence_length) y produce tensores 3D (samples, sequence_length, embedding_dimensionality), la dimension extra es por que a cada indice de palabra se mapea a un embedding.

El diccionario interno de la capa `Embedding` inicialmente esta configurado con valores aleatorios, durante el entrenamiento gradualmente se ajustan via *backpropagation*. Esto estructura los datos en algo que el resto del modelo pueden aprovechar. 

# Embeddings para IMDb prediccion de sentimientos

Se van a calcular embeddings en contexto al problema de predicción de sentimientos de IMDb.

Primero se restringiran las muestras a las 10000 palabras mas comunes y se truncaran a un maximo de 20 palabras. En la capa `Embeddings` se entrenara un embedding para cada una de las 10000 palabras (interpretadas como indices enteros). 

In [2]:
from keras.datasets import imdb
from keras import preprocessing

max_features = 1000
maxlen = 20

(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=max_features)

print('x_train[0]:', x_train[0]) # Reviews codificadas como enteros
print('x_train[0]:', x_test[0])


x_train = preprocessing.sequence.pad_sequences(x_train, maxlen=maxlen) # Trunca cada ejemplo a 20 palabras (por default las 20 ultimas)
x_test = preprocessing.sequence.pad_sequences(x_test, maxlen=maxlen)

print('x_train[0]:', x_train[0])
print('x_train[0]:', x_test[0])


x_train[0]: [1, 14, 22, 16, 43, 530, 973, 2, 2, 65, 458, 2, 66, 2, 4, 173, 36, 256, 5, 25, 100, 43, 838, 112, 50, 670, 2, 9, 35, 480, 284, 5, 150, 4, 172, 112, 167, 2, 336, 385, 39, 4, 172, 2, 2, 17, 546, 38, 13, 447, 4, 192, 50, 16, 6, 147, 2, 19, 14, 22, 4, 2, 2, 469, 4, 22, 71, 87, 12, 16, 43, 530, 38, 76, 15, 13, 2, 4, 22, 17, 515, 17, 12, 16, 626, 18, 2, 5, 62, 386, 12, 8, 316, 8, 106, 5, 4, 2, 2, 16, 480, 66, 2, 33, 4, 130, 12, 16, 38, 619, 5, 25, 124, 51, 36, 135, 48, 25, 2, 33, 6, 22, 12, 215, 28, 77, 52, 5, 14, 407, 16, 82, 2, 8, 4, 107, 117, 2, 15, 256, 4, 2, 7, 2, 5, 723, 36, 71, 43, 530, 476, 26, 400, 317, 46, 7, 4, 2, 2, 13, 104, 88, 4, 381, 15, 297, 98, 32, 2, 56, 26, 141, 6, 194, 2, 18, 4, 226, 22, 21, 134, 476, 26, 480, 5, 144, 30, 2, 18, 51, 36, 28, 224, 92, 25, 104, 4, 226, 65, 16, 38, 2, 88, 12, 16, 283, 5, 16, 2, 113, 103, 32, 15, 16, 2, 19, 178, 32]
x_train[0]: [1, 591, 202, 14, 31, 6, 717, 10, 10, 2, 2, 5, 4, 360, 7, 4, 177, 2, 394, 354, 4, 123, 9, 2, 2, 2, 10, 10


Cada indice de palabra se agrupa en una secuencia eso se ingresa a la capa `Embedding` y esta mapea a un embeding que es usado como input del resto de la red Densa. Todo esto se procesa de a lotes de muestras, por lo que el mapeo es tal que asi:

$$
2D (samples, seq_len) -> Embedding -> 3D (samples, seq_len, emb_len) -> Clasificador Denso -> Prediccion (0 | 1)
$$

In [ ]:
from keras.models import Sequential
from keras.layers import Flatten, Dense

model = Sequential()
model.add(Embedding(10000, 8, input_length=maxlen)) # 10000 posibles tokens, len(vector) = 8

model.add(Flatten())    # La salida de Embedding es 3D, se aplana a 2D (samples, maxlen * 8)

model.add(Dense(1, activation='sigmoid'))   # Clasificador

model.compile(optimizer='rmsprop', loss='binary_crossentropy', metrics=['acc'])
model.summary()

history = model.fit(x_train, y_train, epochs=10, batch_size=32, validation_split=0.2)

I0000 00:00:1755995358.082290   39438 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 11528 MB memory:  -> device: 0, name: NVIDIA TITAN X (Pascal), pci bus id: 0000:01:00.0, compute capability: 6.1


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

...

Epoch 1/10


I0000 00:00:1755995359.395084   39757 service.cc:152] XLA service 0x7f5ff4004e60 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1755995359.395167   39757 service.cc:160]   StreamExecutor device (0): NVIDIA TITAN X (Pascal), Compute Capability 6.1
2025-08-23 21:29:19.989902: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1755995360.810723   39757 cuda_dnn.cc:529] Loaded cuDNN version 90300


 91/625 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - acc: 0.5067 - loss: 0.6931

I0000 00:00:1755995365.543047   39757 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


625/625 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - acc: 0.5620 - loss: 0.6857 - val_acc: 0.6846 - val_loss: 0.6237
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - acc: 0.7175 - loss: 0.5847 - val_acc: 0.7106 - val_loss: 0.5510
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - acc: 0.7449 - loss: 0.5113 - val_acc: 0.7252 - val_loss: 0.5316
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - acc: 0.7641 - loss: 0.4869 - val_acc: 0.7314 - val_loss: 0.5291
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - acc: 0.7714 - loss: 0.4774 - val_acc: 0.7320 - val_loss: 0.5300
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - acc: 0.7718 - loss: 0.4688 - val_acc: 0.7310 - val_loss: 0.5298
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - acc: 0.7893 - loss: 0.4511 - val_acc: 0.7304 - val_loss: 0.5332
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - acc: 0.7849 - loss: 0.4503 - val_acc: 0.7286 - val_loss: 0.5356
Epoch 9/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - acc: 0.7934 -

In [8]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ (32, 20, 8)            │        80,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (32, 160)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (32, 1)                │           161 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 160,324 (626.27 KB)

 Trainable params: 80,161 (313.13 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 80,163 (313.14 KB)

El acc es de ~73%, es un buen valor si se considera que solo se estan mirando 20 palabras en cada review.

El modelo anterior meramente aplana la entrada y la pasa por 1 sola capa Densa por lo que **no considera** las relaciones entre palabras ni la estructura de las oraciones, este modelo tomaria las frases *"This movie is a bomb"* y *"This movie is the bomb"* ambas como criticas negativas.

Añadir capas a continuacion de la capa `Embedding` puede permitir que consideren las relaciones entre palabras y la semantica.

# Embeddings 

Se pueden aplicar embeddings preentrenados a un modelo de forma similar a como se aplicaban las *convnets* preentrenadas en los modelo de vision por computadora.

En concreto 2 fuentes de embeddings preentrenados son:
- [Word2vec](https://code.google.com/archive/p/word2vec/)
- [GloVe](https://nlp.stanford.edu/projects/glove/)